In [ ]:
import sys 
sys.path.append('..')

import pickle
import tarfile

import pandas as pd

from scripts.preprocessing.utils import download_nltk_files, preprocess_text
from scripts.preprocessing.visualization import plot_text_distribution, plot_yearly_publications

In [ ]:
download_nltk_files()

---
### **STAGE 1. Abstract Dataset Extraction**
---

In [ ]:
df = pd.read_csv("../data/raw/processed_articles_real.csv", engine='python')
df = df.dropna()
df

---
### **STAGE 2. Corpus Preprocessing**
---

In [ ]:
def refactor_countries(row):
    countries = row["country"]
    countries = countries.strip("{}").split(",")
        
    return str(countries)


def refactor_authors(row):
    authors = row["authors"]
    authors = authors.strip("{}").split(",")
    authors = [i.replace('"', '').replace("'", "").strip() for i in authors]
        
    return str(authors)


def refactor_institutions(row):
    institutions = row["institution"]
    institutions = institutions.strip("{}").split(",")
    institutions = [i.replace('"', '').replace("'", "").strip() for i in institutions]
        
    return str(institutions)


def refactor_citations(row):
    citations = row["citations"]
    citations = citations.strip("{}").split(",")
    citations = [int(i) for i in citations if i.isdigit()]
        
    return str(citations)

In [ ]:
from joblib import Parallel, delayed

processed_texts = Parallel(n_jobs=16, backend="multiprocessing")(
    delayed(preprocess_text)(text, lemmatize=True) for text in df["text"]
)

df["processed_text"] = processed_texts
df["authors"] = df.apply(refactor_authors, axis=1)
df["country"] = df.apply(refactor_countries, axis=1)
df["citations"] = df.apply(refactor_citations, axis=1)
df["institution"] = df.apply(refactor_institutions, axis=1)
df["publication_year"] = pd.to_datetime(df["publication_date"], errors='coerce').dt.year
df["citation_count"] = df["citations"].apply(lambda x: len(eval(x)) if pd.notnull(x) else 0)

In [ ]:
df

In [ ]:
df.to_csv('../data/corpus/topic_dataset.csv', index=False)

---
### **STAGE 3. Result Visualization**
---

In [ ]:
plot_text_distribution(df, "text")

In [ ]:
plot_text_distribution(df, "processed_text")

In [ ]:
pubs_per_year = df['publication_year'].value_counts().sort_index()

plot_yearly_publications(pubs_per_year)